# 4.0 — Setup do SALib

Notebook de validação de ambiente para a Etapa 4 (Análise de Sensibilidade Global via Sobol).  
Nenhum resultado é salvo — o propósito é detectar problemas antes das sub-etapas 4.1–4.4.

**Checklist:**
1. SALib importa sem erro
2. Definição de `problem_8` (8 inputs) e `problem_6` (6 inputs)
3. Carregamento dos modelos SVR baseline (Etapa 2) e reduzido k\*=6 (Etapa 3)
4. Carregamento dos scalers de X
5. Teste de amostragem Saltelli e predição

## 1 — Importações e versão do SALib

In [ ]:
import importlib.metadata
import numpy as np
import joblib
import pathlib

from SALib.sample import saltelli
from SALib.analyze import sobol

salib_version = importlib.metadata.version("SALib")
print(f"SALib version : {salib_version}")
print(f"numpy version : {np.__version__}")
print(f"joblib version: {joblib.__version__}")
print("✓ Todas as importações concluídas sem erro.")

## 2 — Definição dos problemas de Sobol

In [ ]:
# Faixas físicas conforme CLAUDE.md
problem_8 = {
    "num_vars": 8,
    "names":    ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"],
    "bounds":   [
        [50, 100],      # P1   — pressão do reator (bar)
        [200, 300],     # T1   — temperatura do reator (°C)
        [85, 95],       # T2   — temperatura entrada coluna (°C)
        [1, 10],        # RRC1 — razão de refluxo coluna 1
        [0.5, 10],      # BRC1 — razão de boil-up coluna 1
        [1, 10],        # RRC2 — razão de refluxo coluna 2
        [0.5, 10],      # BRC2 — razão de boil-up coluna 2
        [0.01, 0.25],   # RFF  — fração de purga
    ],
}

# S_6 = {T1, RRC1, BRC1, RRC2, BRC2, RFF} — features selecionadas em k*=6
problem_6 = {
    "num_vars": 6,
    "names":    ["T1", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"],
    "bounds":   [
        [200, 300],
        [1, 10],
        [0.5, 10],
        [1, 10],
        [0.5, 10],
        [0.01, 0.25],
    ],
}

# Índices das 6 features dentro do vetor de 8 (usados para fatiar o scaler)
IDX_S6 = [1, 3, 4, 5, 6, 7]  # T1, RRC1, BRC1, RRC2, BRC2, RFF

print("problem_8:", problem_8["names"])
print("problem_6:", problem_6["names"])
print(f"IDX_S6   : {IDX_S6}")
print("\u2713 Problemas definidos.")

## 3 — Carregamento dos modelos SVR

In [ ]:
BASE = pathlib.Path("../../")  # raiz do projeto a partir de ARTEFATOS/ETAPA_4/

PATHS_BASELINE = {
    "ET":      BASE / "ARTEFATOS/ETAPA_2/SVR/ET/model.pkl",
    "M_CH3OH": BASE / "ARTEFATOS/ETAPA_2/SVR/M_CH3OH/model.pkl",
    "x_CH3OH": BASE / "ARTEFATOS/ETAPA_2/SVR/x_CH3OH/model.pkl",
}

PATHS_REDUZIDO = {
    "ET":      BASE / "ARTEFATOS/ETAPA_3/reduzido/SVR/ET/k6/model.pkl",
    "M_CH3OH": BASE / "ARTEFATOS/ETAPA_3/reduzido/SVR/M_CH3OH/k6/model.pkl",
    "x_CH3OH": BASE / "ARTEFATOS/ETAPA_3/reduzido/SVR/x_CH3OH/k6/model.pkl",
}

models_baseline = {}
models_reduzido = {}

for output, path in PATHS_BASELINE.items():
    assert path.exists(), f"Arquivo não encontrado: {path}"
    models_baseline[output] = joblib.load(path)
    print(f"  [baseline] {output}: carregado ({type(models_baseline[output]).__name__})")

for output, path in PATHS_REDUZIDO.items():
    assert path.exists(), f"Arquivo não encontrado: {path}"
    models_reduzido[output] = joblib.load(path)
    print(f"  [reduzido] {output}: carregado ({type(models_reduzido[output]).__name__})")

print("\u2713 Todos os modelos SVR carregados.")

## 4 — Carregamento dos scalers

In [ ]:
PROCESSED = BASE / "ARTEFATOS/ETAPA_0/processed"

scaler_X_min   = np.load(PROCESSED / "scaler_X_min.npy")    # shape (8,)
scaler_X_scale = np.load(PROCESSED / "scaler_X_scale.npy")  # shape (8,)

assert scaler_X_min.shape == (8,),   f"shape inesperado: {scaler_X_min.shape}"
assert scaler_X_scale.shape == (8,), f"shape inesperado: {scaler_X_scale.shape}"
assert np.all(scaler_X_scale > 0),   "scaler_X_scale contém zeros — divisão por zero!"

print(f"scaler_X_min   shape: {scaler_X_min.shape}  | valores: {np.round(scaler_X_min, 4)}")
print(f"scaler_X_scale shape: {scaler_X_scale.shape} | valores: {np.round(scaler_X_scale, 4)}")

# Função de normalização — reutilizada em 4.1 e 4.2
def normalize_X(X_phys, min_=scaler_X_min, scale_=scaler_X_scale):
    """Transformação min-max idêntica à usada no treinamento."""
    return (X_phys - min_) / scale_

print("\u2713 Scalers carregados e válidos.")

## 5 — Teste de amostragem Saltelli + predição

In [ ]:
N_TEST = 8  # amostragem mínima — só para validação do pipeline

# --- problem_8 ---
samp_8 = saltelli.sample(problem_8, N_TEST, calc_second_order=False)
expected_rows_8 = N_TEST * (problem_8["num_vars"] + 2)
assert samp_8.shape == (expected_rows_8, 8), f"shape inesperado: {samp_8.shape}"
print(f"problem_8 — Saltelli shape: {samp_8.shape}  (esperado: {(expected_rows_8, 8)})")

samp_8_norm = normalize_X(samp_8)
for output, model in models_baseline.items():
    Y = model.predict(samp_8_norm)
    assert Y.shape == (expected_rows_8,), f"shape de Y inesperado: {Y.shape}"
    print(f"  baseline/{output}: predict OK — shape={Y.shape}, min={Y.min():.4f}, max={Y.max():.4f}")

# --- problem_6 ---
samp_6 = saltelli.sample(problem_6, N_TEST, calc_second_order=False)
expected_rows_6 = N_TEST * (problem_6["num_vars"] + 2)
assert samp_6.shape == (expected_rows_6, 6), f"shape inesperado: {samp_6.shape}"
print(f"\nproblem_6 — Saltelli shape: {samp_6.shape}  (esperado: {(expected_rows_6, 6)})")

samp_6_norm = normalize_X(samp_6, min_=scaler_X_min[IDX_S6], scale_=scaler_X_scale[IDX_S6])
for output, model in models_reduzido.items():
    Y = model.predict(samp_6_norm)
    assert Y.shape == (expected_rows_6,), f"shape de Y inesperado: {Y.shape}"
    print(f"  reduzido/{output}: predict OK — shape={Y.shape}, min={Y.min():.4f}, max={Y.max():.4f}")

print("\n✓ Amostragem Saltelli e predição funcionando corretamente.")

## Resultado Final

In [ ]:
print("=" * 50)
print("RESUMO DO SETUP — ETAPA 4.0")
print("=" * 50)
print(f"SALib             : {salib_version}")
print(f"Models baseline   : {list(models_baseline.keys())}")
print(f"Models reduzido   : {list(models_reduzido.keys())}")
print(f"scaler_X_min shape: {scaler_X_min.shape}")
print(f"IDX_S6 (k*=6)     : {IDX_S6}")
print(f"problem_8 inputs  : {problem_8['names']}")
print(f"problem_6 inputs  : {problem_6['names']}")
print("=" * 50)
print("TUDO OK — ambiente validado para Etapa 4.")